In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

# Load API key from .env file into environment
load_dotenv()

# Initialize the Anthropic client (reads ANTHROPIC_API_KEY from env automatically)
client = Anthropic()
# Use Haiku for fast, cost-effective responses
model = "claude-haiku-4-5"

In [ ]:
# Helper functions

# Appends a user message dict to the conversation history
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

# Appends an assistant message dict to the conversation history
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Sends conversation to Claude and returns the text response
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [4]:
from anthropic.types import ToolParam

# Step 1 - Write a tool function 
# Step 2 - Write a JSON schema
# Step 3 - Call Claude with JSON schema
# Step 4 - Run Tool 
# Step 5 - Add Tool Result block and call Claude again

In [ ]:
# Step 1 - Write a tool function 
def celsius_to_temperature(celsius: float, unit: str = "fahrenheit") -> str:
    """
    Converts a Celsius temperature to the specified unit.
    
    Args:
        celsius: Temperature value in Celsius.
        unit: Target unit — "fahrenheit" or "kelvin". Defaults to "fahrenheit".

    Returns:
        A formatted string like "72.5 °F" or "310.15 K".

    Raises:
        ValueError: If unit is not "fahrenheit" or "kelvin".
        ValueError: If celsius is below absolute zero (-273.15).
    """
    VALID_UNITS = {"fahrenheit", "kelvin"}
    
    # Validate the target unit
    if unit.lower() not in VALID_UNITS:
        raise ValueError(f"unit must be one of {VALID_UNITS}, got '{unit}'")
    
    # Reject physically impossible temperatures
    if celsius < -273.15:
        raise ValueError(f"celsius cannot be below absolute zero (-273.15), got {celsius}")
    
    # Apply the conversion formula based on target unit
    if unit.lower() == "fahrenheit":
        result = (celsius * 9/5) + 32
        return f"{result:.2f} °F"
    else:
        result = celsius + 273.15
        return f"{result:.2f} K"

In [ ]:
# Step 2 - Write a JSON schema describing the tool for Claude
# ToolParam adds type validation on top of a plain dict
celsius_to_temperature_schema = ToolParam({
  "name": "celsius_to_temperature",
  "description": "Converts a Celsius temperature value to either Fahrenheit or Kelvin. Returns a formatted string with the result and unit symbol, e.g. '72.50 °F' or '310.15 K'. The input must be a physically valid temperature (no lower than absolute zero, -273.15 °C).",
  "input_schema": {
    "type": "object",
    "properties": {
      "celsius": {
        "type": "number",
        "description": "The temperature in Celsius to convert. Must be >= -273.15 (absolute zero). Examples: 0 (water freezing point), 100 (water boiling point), -40 (where Celsius and Fahrenheit coincide).",
        "minimum": -273.15
      },
      "unit": {
        "type": "string",
        "description": "The target temperature unit for conversion. Use 'fahrenheit' for °F (common in the US) or 'kelvin' for K (used in scientific contexts). Case-insensitive.",
        "enum": ["fahrenheit", "kelvin"],
        "default": "fahrenheit"
      }
    },
    "required": ["celsius"]
  }
}
)

In [ ]:
# Step 3.1 - Call Claude with the tool schema
# Send user prompt + tool definition so Claude knows it can call celsius_to_temperature
messages = []

messages.append(
    {
        "role": "user",
        "content": "What is 11°C in Fahrenheit?  Guidelines: Reply only with the value, no additional text"
    }
)

# Claude will respond with a tool_use block instead of a text answer
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[celsius_to_temperature_schema]
)

# stop_reason will be "tool_use" since Claude wants to call the tool
print(f"Stop reason: {response.stop_reason}") 

response

In [ ]:
# Step 3.2 - Extract Claude's tool call details from the response
# Find the tool_use block in the response content
tool_use_block = next(b for b in response.content if b.type == "tool_use")
tool_name = tool_use_block.name          # → "celsius_to_temperature"
tool_input = tool_use_block.input        # → {"celsius": 11, "unit": "fahrenheit"}
tool_call_id = tool_use_block.id         # Unique ID to link the result back

print(f"Tool first call status: \n tool_name = {tool_name} \n tool_input = {tool_input} \n tool_call_id = {tool_call_id}")

In [ ]:
# Step 4 - Run Tool
# Execute the actual Python function with the arguments Claude provided
tool_result = celsius_to_temperature(**tool_input)  # → "51.80 °F"

tool_result

In [ ]:
# Step 5 - Add Tool Result block and call Claude again
# First, add Claude's original response (with the tool_use block) to the conversation
messages.append(
{
    "role":"assistant",
    "content": response.content
})

# Then send the tool result back as a user message with type "tool_result"
messages.append(
{
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_call_id,  # Must match the tool_use block's ID
            "content": tool_result,        # The actual output from running the function
            "is_error": False
        }
    ]
})

messages

# Final call: Claude now has the tool result and can give a natural language answer
final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[celsius_to_temperature_schema]
)

print(final_response.content[0].text)